In [ ]:
# Faith Yang & Robert Armenta (Monday/Wednesday Lab)
# Polynomial Model
# Dataset: https://www.kaggle.com/datasets/abbas829/bank-customer-churn

In [ ]:
# Import necessary libraries
import pandas as pd              # Data handling (tables, CSV files)
import numpy as np               # Numerical operations

from sklearn.model_selection import train_test_split  # Split data into train/test
from sklearn.svm import SVC      # Support Vector Machine model
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report  # Evaluation
import matplotlib.pyplot as plt  # Plotting graphs
import seaborn as sns            # Better visualization (heatmaps, etc.)
from sklearn.preprocessing import StandardScaler, LabelEncoder  # Scaling + encoding

In [ ]:
# load dataset
df = pd.read_csv("Bank_Churn.csv")
df.head()

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
# dataset structure overiew including data types and missing values
df.info()
print(df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  int64  
 2   Gender           10000 non-null  int64  
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9)
memory usage: 859.5 KB
(10000, 11)


In [ ]:
# Data cleaning
# Check missing values
print("Missing values:")
print(df.isnull().sum())
print()

# Drop missing values
df = df.dropna()

print("Missing values removed.")
print()

# Drop unnecessary columns
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"], errors='ignore')

# Encode categorical columns
label_encoders = {}

categorical_columns = df.select_dtypes(include=["object"]).columns

for col in categorical_columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

Missing values:
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

Missing values removed.



In [ ]:
# number of unique values in each columnn
df.nunique()

,0
CreditScore,460
Geography,3
Gender,2
Age,70
Tenure,11
Balance,6382
NumOfProducts,4
HasCrCard,2
IsActiveMember,2
EstimatedSalary,9999


In [ ]:
# display all column names in the dataset
print(df.columns.tolist())

['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited']


In [ ]:
# Define features (X) and target (y)
X = df.drop("Exited", axis=1)   # Features
y = df["Exited"]                # Target (0 = stayed, 1 = churned)

In [ ]:
# Split data into training (70%) and testing (30%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# SVM Polynomial Model
svm_poly = SVC(kernel='poly', degree=3, C=1.0, gamma='scale')
svm_poly.fit(X_train, y_train)

SVC(kernel='poly')

In [ ]:
# Predictions
y_pred = svm_poly.predict(X_test)

In [ ]:
# Model Evaluation (Reliability)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Accuracy: 85.87%

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.99      0.92      2416
           1       0.87      0.32      0.47       584

    accuracy                           0.86      3000
   macro avg       0.87      0.65      0.69      3000
weighted avg       0.86      0.86      0.83      3000


Confusion Matrix:
[[2389   27]
 [ 397  187]]


In [ ]:
# Support vectors
print(f"\nNumber of Support Vectors: {len(svm_poly.support_vectors_)}")


Number of Support Vectors: 2459


In [ ]:
# New Data Entry (Prediction)
# Example new customer:
# Features expected by the model:
# ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
# Geography=0 (France) and Gender=0 (Female) as placeholders.
new_customer = [
    600,  # CreditScore
    0,    # Geography (e.g., 0 for France, 1 for Germany, 2 for Spain, based on LabelEncoder)
    0,    # Gender (e.g., 0 for Female, 1 for Male, based on LabelEncoder)
    40,   # Age
    3,    # Tenure
    60000,# Balance
    2,    # NumOfProducts
    1,    # HasCrCard
    1,    # IsActiveMember
    50000 # EstimatedSalary
]

new_customer_df = pd.DataFrame([new_customer], columns=X.columns)
new_customer_scaled = scaler.transform(new_customer_df)

prediction = svm_poly.predict(new_customer_scaled)

print("\nNew Customer Prediction:")
if prediction[0] == 1:
    print("This customer is likely to CHURN")
else:
    print("This customer is likely to STAY")


New Customer Prediction:
This customer is likely to STAY


This project uses a Support Vector Machine (SVM) with a polynomial kernel to predict whether a bank customer will leave the bank or stay. The polynomial kernel allows the model to create curved decision boundaries, which helps capture more complex patterns in customer behavior. After training the model using historical data, we tested it on new data and evaluated its performance using accuracy, confusion matrix, and classification report. The results show how well the model can correctly identify customers who are likely to churn. We also tested a new customer example, showing how the model can be used in real-world situations to help businesses reduce customer loss.